# AMDT — Multi-Dataset AI-Image Detector (Kaggle)

Full pipeline: dataset download → preparation → model → training → evaluation.

**Requirements:**
- GPU: T4 x2 or P100
- Internet: On (Settings → Internet → On)
- Kaggle Secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`

In [ ]:
import os, sys
print("Python :", sys.version.split()[0])
print("CPUs   :", os.cpu_count())
import torch
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
!pip install -q timm einops scikit-learn
import timm, einops
print("timm:", timm.__version__)
print("einops:", einops.__version__)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = user_secrets.get_secret("KAGGLE_KEY")
print("Kaggle credentials set.")

In [ ]:
import os, glob

RAW_DIR = "/kaggle/working/raw"
os.makedirs(RAW_DIR, exist_ok=True)

DOWNLOADS = [
    ("birdy654/cifake-real-and-ai-generated-synthetic-images", "CIFAKE",                    110),
    ("alessandrasala79/ai-vs-human-generated-dataset",         "AI-vs-Human",              2400),
    ("xhlulu/140k-real-and-fake-faces",                        "140k-Real-and-Fake-Faces", 800),
]

def _collect(d):
    exts = ("*.jpg", "*.jpeg", "*.png", "*.webp")
    out = []
    for e in exts:
        out += glob.glob(os.path.join(d, "**", e), recursive=True)
    return out

for slug, name, size_mb in DOWNLOADS:
    out_dir = os.path.join(RAW_DIR, name)
    os.makedirs(out_dir, exist_ok=True)
    existing = _collect(out_dir)
    if len(existing) > 100:
        print(f"[{name}] Already present ({len(existing):,} images). Skipping.")
        continue
    print(f"[{name}] Downloading ~{size_mb} MB ...")
    ret = os.system(f"kaggle datasets download -d {slug} -p {out_dir} --unzip -q")
    if ret != 0:
        print(f"[{name}] FAILED")
    else:
        print(f"[{name}] Done — {len(_collect(out_dir)):,} images")

In [ ]:
import os, glob, shutil, random, logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s")
log = logging.getLogger(__name__)

DATA_DIR = "/kaggle/working/data"

def _collect(src_dirs, exts=("*.jpg", "*.jpeg", "*.png", "*.webp")):
    out = []
    for d in src_dirs:
        for ext in exts:
            out += glob.glob(os.path.join(d, "**", ext), recursive=True)
    return out

def _copy_hashed(paths, dst, max_n=None, seed=42):
    os.makedirs(dst, exist_ok=True)
    if max_n and len(paths) > max_n:
        rng = random.Random(seed)
        paths = rng.sample(paths, max_n)
    n = 0
    for p in paths:
        fname = f"{abs(hash(p)) % (10**10)}_{os.path.basename(p)}"
        try:
            shutil.copy2(p, os.path.join(dst, fname))
            n += 1
        except Exception:
            pass
    return n

def normalise_cifake():
    log.info("[CIFAKE] ...")
    raw = os.path.join(RAW_DIR, "CIFAKE")
    for split in ("train", "test"):
        for label in ("REAL", "FAKE"):
            src = os.path.join(raw, split, label)
            dst = os.path.join(DATA_DIR, "cifake", split, label)
            if not os.path.isdir(src):
                log.warning(f"  missing {src}")
                continue
            n = _copy_hashed(_collect([src]), dst)
            log.info(f"  {split}/{label}: {n:,}")

def normalise_aivsreal():
    log.info("[AI-vs-Human] ...")
    raw = os.path.join(RAW_DIR, "AI-vs-Human")
    pairs = [
        ("train", os.path.join(raw, "train_data",   "real"), os.path.join(raw, "train_data",   "fake")),
        ("test",  os.path.join(raw, "test_data_v2", "real"), os.path.join(raw, "test_data_v2", "fake")),
    ]
    for split, real_src, fake_src in pairs:
        if not os.path.isdir(fake_src):
            alt = fake_src.replace("fake", "ai_generated")
            if os.path.isdir(alt):
                fake_src = alt
        for label, src in [("REAL", real_src), ("FAKE", fake_src)]:
            if not os.path.isdir(src):
                log.warning(f"  missing {src}")
                continue
            dst = os.path.join(DATA_DIR, "aivsreal", split, label)
            n = _copy_hashed(_collect([src]), dst, max_n=30000)
            log.info(f"  {split}/{label}: {n:,}")

def normalise_140k():
    log.info("[140k Faces] ...")
    raw = os.path.join(RAW_DIR, "140k-Real-and-Fake-Faces")
    base = os.path.join(raw, "real_vs_fake", "real-vs-fake")
    if not os.path.isdir(base):
        cands = glob.glob(os.path.join(raw, "**", "real-vs-fake"), recursive=True)
        if cands:
            base = cands[0]
        else:
            log.error("  real-vs-fake not found")
            return
    for split, src_split in (("train", "train"), ("test", "valid")):
        for label, src_label in (("REAL", "real"), ("FAKE", "fake")):
            src = os.path.join(base, src_split, src_label)
            dst = os.path.join(DATA_DIR, "faces140k", split, label)
            if not os.path.isdir(src):
                log.warning(f"  missing {src}")
                continue
            n = _copy_hashed(_collect([src]), dst)
            log.info(f"  {split}/{label}: {n:,}")

normalise_cifake()
normalise_aivsreal()
normalise_140k()

print("\n" + "=" * 62)
print("DATA SUMMARY")
print("=" * 62)
tr = tf = te = tf_ = 0
for ds in ("cifake", "aivsreal", "faces140k"):
    c = {}
    for split in ("train", "test"):
        for label in ("REAL", "FAKE"):
            d = os.path.join(DATA_DIR, ds, split, label)
            n = len(glob.glob(os.path.join(d, "*"))) if os.path.isdir(d) else 0
            c[(split, label)] = n
    print(f"  {ds:<12} | train REAL={c[('train','REAL')]:>7,} FAKE={c[('train','FAKE')]:>7,} "
          f"| test REAL={c[('test','REAL')]:>6,} FAKE={c[('test','FAKE')]:>6,}")
    tr  += c[("train", "REAL")];  tf += c[("train", "FAKE")]
    te  += c[("test",  "REAL")];  tf_ += c[("test", "FAKE")]
print("-" * 62)
print(f"  TOTAL | train REAL={tr:>7,} FAKE={tf:>7,} | test REAL={te:>6,} FAKE={tf_:>6,}")
print(f"  Train REAL/FAKE ratio: {tr/max(tf,1):.3f}")
print("=" * 62)

In [ ]:
import os, glob, time, logging, warnings
import numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import transforms
from PIL import Image
from einops import rearrange
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             confusion_matrix, precision_score, recall_score)
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import timm
warnings.filterwarnings("ignore")

os.makedirs("/kaggle/working/outputs/plots", exist_ok=True)
os.makedirs("/kaggle/working/outputs/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/outputs/logs", exist_ok=True)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)s | %(message)s",
                    handlers=[logging.FileHandler("/kaggle/working/outputs/logs/train.log"),
                              logging.StreamHandler()])
log = logging.getLogger(__name__)

class Config:
    CIFAKE_ROOT    = f"{DATA_DIR}/cifake"
    AIVSREAL_ROOT  = f"{DATA_DIR}/aivsreal"
    FACES_ROOT     = f"{DATA_DIR}/faces140k"
    IMG_SIZE       = 224
    BATCH_SIZE     = 16
    NUM_WORKERS    = 2
    MAX_SAMPLES    = None
    MODEL_NAME     = "vit_base_patch16_224"
    EMBED_DIM      = 768
    NUM_HEADS      = 12
    NUM_LAYERS     = 12
    DROPOUT        = 0.3
    NUM_SRM_KERNELS = 3
    EPOCHS         = 5
    STAGE1_EPOCHS  = 1
    STAGE2_EPOCHS  = 4
    LR             = 2e-4
    BACKBONE_LR    = 2e-5
    WARMUP_EPOCHS  = 1
    WEIGHT_DECAY   = 1e-4
    GRAD_CLIP      = 1.0
    USE_AMP        = True
    LABEL_SMOOTH   = 0.1
    CUTMIX_ALPHA   = 1.0
    PATIENCE       = 3
    SEED           = 42
    DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CHECKPOINT     = "/kaggle/working/outputs/checkpoints/amdt_best.pt"

cfg = Config()
print("Device:", cfg.DEVICE, "| GPUs:", torch.cuda.device_count())

def _amp_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    major, _ = torch.cuda.get_device_capability(0)
    return torch.bfloat16 if major >= 8 else torch.float16

In [ ]:
def _srm_kernels():
    k1 = np.array([[0,0,0,0,0],[0,-1,2,-1,0],[0,2,-4,2,0],[0,-1,2,-1,0],[0,0,0,0,0]], dtype=np.float32)/4.0
    k2 = np.array([[-1,2,-2,2,-1],[2,-6,8,-6,2],[-2,8,-12,8,-2],[2,-6,8,-6,2],[-1,2,-2,2,-1]], dtype=np.float32)/12.0
    k3 = np.array([[0,0,0,0,0],[0,0,0,0,0],[0,1,-2,1,0],[0,0,0,0,0],[0,0,0,0,0]], dtype=np.float32)/2.0
    return [k1, k2, k3]

class SRMFilter(nn.Module):
    def __init__(self, num_kernels=3):
        super().__init__()
        kernels = _srm_kernels()[:num_kernels]
        weights = [torch.FloatTensor(k).unsqueeze(0).unsqueeze(0) for k in kernels for _ in range(3)]
        self.conv = nn.Conv2d(3, len(weights), 5, padding=2, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(torch.cat(weights, dim=0))
        self.conv.weight.requires_grad = False
    def forward(self, x):
        return torch.tanh(self.conv(x))

class FreqNet(nn.Module):
    def __init__(self, channels=3):
        super().__init__()
        self.channels = channels
        self.freq_conv = nn.Sequential(
            nn.Conv2d(channels*2, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, channels*2, 1),
        )
    def forward(self, x):
        orig_dtype = x.dtype
        freq = torch.fft.rfft2(x.to(torch.float32), norm='ortho')
        combined = torch.cat([freq.real, freq.imag], dim=1)
        processed = self.freq_conv(combined).to(torch.float32)
        real, imag = processed[:, :self.channels], processed[:, self.channels:]
        out = torch.fft.irfft2(torch.complex(real, imag), s=x.shape[-2:], norm='ortho')
        return out.to(orig_dtype)

class ForgeryAwareAdapter(nn.Module):
    def __init__(self, dim=768):
        super().__init__()
        self.spatial_adapter = nn.Sequential(
            nn.Conv2d(dim, dim//4, 3, padding=1), nn.GELU(),
            nn.Conv2d(dim//4, dim, 3, padding=1))
        self.freq_adapter = nn.Sequential(
            nn.Conv2d(dim, dim//4, 3, stride=2, padding=1), nn.GELU(),
            nn.ConvTranspose2d(dim//4, dim, 4, stride=2, padding=1))
        self.fusion = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim))
    def forward(self, x):
        B, N, D = x.shape
        cls = x[:, :1]; patches = x[:, 1:]
        H = int(patches.shape[1] ** 0.5); W = patches.shape[1] // H
        spatial = patches.transpose(1, 2).reshape(B, D, H, W)
        s_feat = self.spatial_adapter(spatial); f_feat = self.freq_adapter(spatial)
        combined = (s_feat + f_feat).flatten(2).transpose(1, 2)
        return torch.cat([cls, self.fusion(combined + patches)], dim=1)

class CrossMultiHeadAttention(nn.Module):
    def __init__(self, dim=768, num_heads=12, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads; self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.q_proj = nn.Linear(dim, dim); self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim); self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout); self.norm = nn.LayerNorm(dim)
    def forward(self, s, a):
        B, N, D = s.shape
        Q = rearrange(self.q_proj(s), 'b n (h d) -> b h n d', h=self.num_heads)
        K = rearrange(self.k_proj(a), 'b n (h d) -> b h n d', h=self.num_heads)
        V = rearrange(self.v_proj(a), 'b n (h d) -> b h n d', h=self.num_heads)
        attn = F.softmax((Q @ K.transpose(-2, -1)) * self.scale, dim=-1)
        out = rearrange(self.dropout(attn) @ V, 'b h n d -> b n (h d)')
        return self.norm(s + self.dropout(self.out_proj(out)))

class MoLDGating(nn.Module):
    def __init__(self, num_layers=12, dim=768):
        super().__init__()
        self.gate_network = nn.Sequential(
            nn.Linear(dim, dim//4), nn.ReLU(inplace=True), nn.Dropout(0.1),
            nn.Linear(dim//4, num_layers), nn.Softmax(dim=-1))
    def forward(self, layer_outputs):
        stacked = torch.stack(layer_outputs, dim=1)
        cls_mean = stacked[:, :, 0, :].mean(dim=1)
        gates = self.gate_network(cls_mean).unsqueeze(-1).unsqueeze(-1)
        return (stacked * gates).sum(dim=1)

class ResidualStreamEncoder(nn.Module):
    def __init__(self, in_channels=9, embed_dim=768, img_size=224, patch_size=16):
        super().__init__()
        reduced_size = img_size // 4
        self.num_patches = (reduced_size // patch_size) ** 2
        self.conv_layers = nn.Sequential(
            nn.Conv2d(in_channels, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.MaxPool2d(2))
        self.patch_embed = nn.Conv2d(256, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.conv_layers(x)
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        return x + self.pos_embed[:, :x.shape[1], :]

class FrequencyStreamEncoder(nn.Module):
    def __init__(self, in_channels=3, embed_dim=768, img_size=224, patch_size=16):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        return x + self.pos_embed[:, :x.shape[1], :]

class AMDT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.vit = timm.create_model(cfg.MODEL_NAME, pretrained=True, num_classes=0)
        self.srm = SRMFilter(cfg.NUM_SRM_KERNELS)
        self.freqnet = FreqNet()
        self.residual_encoder = ResidualStreamEncoder(in_channels=cfg.NUM_SRM_KERNELS*3, embed_dim=cfg.EMBED_DIM)
        self.frequency_encoder = FrequencyStreamEncoder(in_channels=3, embed_dim=cfg.EMBED_DIM)
        self.faa_layers = nn.ModuleList([ForgeryAwareAdapter(cfg.EMBED_DIM) for _ in range(cfg.NUM_LAYERS // 2)])
        self.cma_residual  = nn.ModuleList([CrossMultiHeadAttention(cfg.EMBED_DIM, cfg.NUM_HEADS) for _ in range(cfg.NUM_LAYERS)])
        self.cma_frequency = nn.ModuleList([CrossMultiHeadAttention(cfg.EMBED_DIM, cfg.NUM_HEADS) for _ in range(cfg.NUM_LAYERS)])
        self.mold = MoLDGating(cfg.NUM_LAYERS, cfg.EMBED_DIM)
        self.fusion_proj = nn.Sequential(
            nn.LayerNorm(cfg.EMBED_DIM*3),
            nn.Linear(cfg.EMBED_DIM*3, cfg.EMBED_DIM), nn.GELU())
        self.classifier = nn.Sequential(
            nn.LayerNorm(cfg.EMBED_DIM),
            nn.Linear(cfg.EMBED_DIM, 512), nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, 2))
    def forward(self, x):
        srm_out = self.srm(x); freq_out = self.freqnet(x)
        res_tokens = self.residual_encoder(srm_out)
        freq_tokens = self.frequency_encoder(freq_out)
        x_vit = self.vit.patch_embed(x)
        cls_token = self.vit.cls_token.expand(x_vit.shape[0], -1, -1)
        tokens = torch.cat((cls_token, x_vit), dim=1) + self.vit.pos_embed
        tokens = self.vit.pos_drop(tokens)
        layer_outputs = []
        for i in range(self.cfg.NUM_LAYERS):
            if i % 2 == 0 and i // 2 < len(self.faa_layers):
                tokens = self.faa_layers[i // 2](tokens)
            tokens = self.vit.blocks[i](tokens)
            tokens = tokens + self.cma_residual[i](tokens, res_tokens)
            tokens = tokens + self.cma_frequency[i](tokens, freq_tokens)
            layer_outputs.append(tokens)
        fused = self.mold(layer_outputs)
        combined = torch.cat([fused[:, 0], res_tokens[:, 0], freq_tokens[:, 0]], dim=1)
        return self.classifier(self.fusion_proj(combined))

In [ ]:
class BinaryImageDataset(Dataset):
    MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
    def __init__(self, real_dir, fake_dir, img_size=224, augment=False,
                 max_samples=None, name="?"):
        self.samples = []
        for ext in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
            self.samples.extend([(p, 0) for p in glob.glob(os.path.join(real_dir, "**", ext), recursive=True)])
            self.samples.extend([(p, 1) for p in glob.glob(os.path.join(fake_dir, "**", ext), recursive=True)])
        if max_samples and max_samples < len(self.samples):
            rng = np.random.default_rng(42)
            idx = rng.choice(len(self.samples), max_samples, replace=False)
            self.samples = [self.samples[i] for i in idx]
        log.info(f"[{name}] {len(self.samples):,} samples (augment={augment})")
        base = [transforms.Resize((img_size, img_size)), transforms.ToTensor(), transforms.Normalize(self.MEAN, self.STD)]
        if augment:
            aug = [transforms.RandomHorizontalFlip(p=0.5),
                   transforms.ColorJitter(0.2, 0.2, 0.1),
                   transforms.RandomRotation(10),
                   transforms.RandomPerspective(distortion_scale=0.1, p=0.3)]
            self.transform = transforms.Compose([base[0]] + aug + base[1:] + [transforms.RandomErasing(p=0.25)])
        else:
            self.transform = transforms.Compose(base)
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return {"spatial": self.transform(img), "label": torch.tensor(label, dtype=torch.long)}

def build_combined_dataset(cfg, split, augment=True):
    datasets = []
    for root, name in [(cfg.CIFAKE_ROOT, "CIFAKE"), (cfg.AIVSREAL_ROOT, "AI-vs-Human"), (cfg.FACES_ROOT, "140k-Faces")]:
        real_dir = f"{root}/{split}/REAL"; fake_dir = f"{root}/{split}/FAKE"
        if os.path.isdir(real_dir) and os.path.isdir(fake_dir):
            datasets.append(BinaryImageDataset(real_dir, fake_dir, cfg.IMG_SIZE,
                augment=(augment and split == "train"), max_samples=cfg.MAX_SAMPLES, name=name))
    if len(datasets) != 3:
        raise RuntimeError(f"Expected 3 datasets, found {len(datasets)}")
    return ConcatDataset(datasets)

def cutmix(imgs, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = imgs.shape
    idx = torch.randperm(B, device=imgs.device)
    ch, cw = int(H * np.sqrt(1 - lam)), int(W * np.sqrt(1 - lam))
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = max(cx - cw//2, 0); x2 = min(cx + cw//2, W)
    y1 = max(cy - ch//2, 0); y2 = min(cy + ch//2, H)
    mixed = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam

def compute_metrics(labels, preds, probs):
    return {"accuracy": accuracy_score(labels, preds),
            "auc": roc_auc_score(labels, probs),
            "f1": f1_score(labels, preds, zero_division=0),
            "precision": precision_score(labels, preds, zero_division=0),
            "recall": recall_score(labels, preds, zero_division=0),
            "conf_matrix": confusion_matrix(labels, preds)}

def get_loaders(cfg):
    train_ds = build_combined_dataset(cfg, "train", augment=True)
    val_ds   = build_combined_dataset(cfg, "test",  augment=False)
    labels = []
    for ds in train_ds.datasets:
        for _, lbl in ds.samples:
            labels.append(lbl)
    labels = np.array(labels)
    counts = np.bincount(labels, minlength=2)
    ratio = counts[0] / max(counts[1], 1)
    log.info(f"Train class counts: REAL={counts[0]:,} FAKE={counts[1]:,} ratio={ratio:.3f}")
    if not (0.85 <= ratio <= 1.15):
        cw = 1.0 / counts
        sw = torch.from_numpy(cw[labels]).double()
        sampler = WeightedRandomSampler(sw, len(sw), replacement=True)
    else:
        sampler = None
    kw = dict(batch_size=cfg.BATCH_SIZE, num_workers=cfg.NUM_WORKERS,
              pin_memory=True, persistent_workers=(cfg.NUM_WORKERS > 0), drop_last=False)
    return (DataLoader(train_ds, sampler=sampler, shuffle=(sampler is None), **kw),
            DataLoader(val_ds, shuffle=False, **kw))

In [ ]:
def _fmt_time(s):
    h, rem = divmod(int(s), 3600); m, sec = divmod(rem, 60)
    return f"{h}h {m:02d}m {sec:02d}s" if h else f"{m}m {sec:02d}s"

LOG_EVERY = 50

def train(cfg):
    torch.manual_seed(cfg.SEED); np.random.seed(cfg.SEED)
    log.info("Building AMDT model...")
    model = AMDT(cfg).to(cfg.DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    log.info(f"AMDT: {n_params:.2f}M params")
    backbone_params = list(model.vit.parameters())
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in model.parameters() if id(p) not in backbone_ids and p.requires_grad]
    optimizer = optim.AdamW([
        {"params": backbone_params, "lr": cfg.BACKBONE_LR},
        {"params": head_params,     "lr": cfg.LR},
    ], weight_decay=cfg.WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTH)
    scaler = torch.amp.GradScaler("cuda", enabled=cfg.USE_AMP)
    amp_dtype = _amp_dtype()
    def lr_lambda(ep):
        if ep < cfg.WARMUP_EPOCHS: return (ep + 1) / cfg.WARMUP_EPOCHS
        p = (ep - cfg.WARMUP_EPOCHS) / max(1, cfg.EPOCHS - cfg.WARMUP_EPOCHS)
        return 0.5 * (1 + np.cos(np.pi * p))
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    log.info("Building DataLoaders...")
    train_loader, val_loader = get_loaders(cfg)
    log.info(f"train={len(train_loader):,} batches  val={len(val_loader):,} batches")
    best_auc = 0.0; patience_left = cfg.PATIENCE
    history = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
    job_start = time.time()
    for epoch in range(cfg.EPOCHS):
        stage = "stage1" if epoch < cfg.STAGE1_EPOCHS else "stage2"
        freeze_backbone = (stage == "stage1")
        for p in model.vit.parameters():
            p.requires_grad = not freeze_backbone
        log.info(f"Epoch {epoch+1}/{cfg.EPOCHS} [{stage}] backbone_frozen={freeze_backbone}")
        epoch_start = time.time()
        model.train(); run_loss = correct = total = 0
        for batch_idx, batch in enumerate(train_loader):
            imgs = batch["spatial"].to(cfg.DEVICE, non_blocking=True)
            labels = batch["label"].to(cfg.DEVICE, non_blocking=True)
            use_cm = np.random.rand() < 0.5
            if use_cm:
                imgs, la, lb, lam = cutmix(imgs, labels, cfg.CUTMIX_ALPHA)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", amp_dtype, enabled=cfg.USE_AMP):
                logits = model(imgs)
                if use_cm:
                    loss = lam * criterion(logits, la) + (1 - lam) * criterion(logits, lb)
                else:
                    loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            run_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)
            if (batch_idx + 1) % LOG_EVERY == 0 or (batch_idx + 1) == len(train_loader):
                done = batch_idx + 1
                spb = (time.time() - epoch_start) / done
                log.info(f"  [Train] Ep{epoch+1:02d} {done:04d}/{len(train_loader):04d} "
                         f"| loss {run_loss/done:.4f} | acc {correct/total*100:.2f}% "
                         f"| {spb:.2f}s/b | ETA {_fmt_time((len(train_loader)-done)*spb)}")
        scheduler.step()
        log.info(f"Train done in {_fmt_time(time.time()-epoch_start)} | avg_loss={run_loss/len(train_loader):.4f}")
        model.eval()
        all_labels, all_preds, all_probs = [], [], []
        val_loss = 0.0; val_start = time.time()
        with torch.no_grad():
            for batch in val_loader:
                imgs = batch["spatial"].to(cfg.DEVICE, non_blocking=True)
                labels = batch["label"].to(cfg.DEVICE, non_blocking=True)
                with torch.autocast("cuda", amp_dtype, enabled=cfg.USE_AMP):
                    logits = model(imgs)
                val_loss += criterion(logits, labels).item()
                all_probs.extend(F.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        m = compute_metrics(all_labels, all_preds, all_probs)
        val_loss /= len(val_loader)
        history["train_loss"].append(run_loss / len(train_loader))
        history["val_loss"].append(val_loss)
        history["val_auc"].append(m["auc"])
        history["val_f1"].append(m["f1"])
        log.info(f"Val: AUC={m['auc']:.4f} F1={m['f1']:.4f} Acc={m['accuracy']*100:.2f}% Loss={val_loss:.4f}")
        log.info(f"Confusion:\n{m['conf_matrix']}")
        if m["auc"] > best_auc:
            best_auc = m["auc"]; patience_left = cfg.PATIENCE
            torch.save({"state_dict": model.state_dict(), "best_auc": best_auc}, cfg.CHECKPOINT)
            log.info(f"[SAVED] best AUC={best_auc:.4f}")
        else:
            patience_left -= 1
            log.info(f"[PATIENCE] {patience_left}/{cfg.PATIENCE}")
            if patience_left == 0:
                log.info("Early stopping."); break
    log.info("=" * 70)
    log.info(f"DONE | best_auc={best_auc:.4f} | total={_fmt_time(time.time()-job_start)}")
    log.info("=" * 70)
    return model, history

In [ ]:
model, history = train(cfg)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True)
axes[1].plot(history["val_auc"], color="green")
axes[1].set_title("Val AUC"); axes[1].grid(True)
axes[2].plot(history["val_f1"], color="orange")
axes[2].set_title("Val F1"); axes[2].grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/outputs/plots/training_history.png", dpi=150, bbox_inches="tight")
plt.show()